# 06 · Muestra de 30 registros para el ejercicio a mano
Se escogen 30 registros al azar (semilla 2026) entre los 5.230 elegibles y se guardan en `datos/muestra_30.csv`.
El ejercicio se resuelve a mano; este notebook solo sirve para comprobar los resultados.

In [1]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import config

el = config.leer("elegibles")
m = el.sample(30, random_state=config.SEED).reset_index(drop=True)
si_no = {True: "Sí", False: "No"}
t = pd.DataFrame({"n": range(1, 31), "mes": m.FECHA_ACTUALIZACION, "X_antiguedad_meses": m.ANTIGUEDAD_MESES.astype(int),
                  "Y_cargo_fijo_cop": m.VAL_CFM.astype(int), "banca": m.BAN_BANCA.map(si_no),
                  "billetera": m.BAN_BILLETERA.map(si_no), "pagos_online": m.BAN_PAGOS_ONLINE.map(si_no),
                  "acepta_mensajes": (~m.BAN_SUPRESION_I.astype(bool)).map(si_no)})
t.to_csv(config.DATOS / "muestra_30.csv", index=False, encoding="utf-8-sig")
t

,n,mes,X_antiguedad_meses,Y_cargo_fijo_cop,banca,billetera,pagos_online,acepta_mensajes
0,1,2026-08,1,40185,No,No,No,No
1,2,2026-07,1,40185,No,No,No,No
2,3,2026-04,4,31879,No,No,No,No
3,4,2026-05,0,40185,No,No,No,No
4,5,2026-03,6,34396,No,No,No,No
5,6,2026-02,4,34396,No,No,No,No
6,7,2026-08,1,40185,No,No,No,No
7,8,2026-07,1,40185,No,No,No,No
8,9,2026-03,6,20221,Sí,Sí,Sí,No
9,10,2026-02,5,42701,No,No,No,No


In [2]:
x, y, n = t.X_antiguedad_meses.to_numpy(), t.Y_cargo_fijo_cop.to_numpy().astype(float), 30
ys = np.sort(y)


def cuartil(k):                      # posición k(n + 1)/4 con interpolación, como en el informe
    pos = k * (n + 1) / 4
    i = int(pos)
    return ys[i - 1] + (pos - i) * (ys[i] - ys[i - 1])


media = y.mean()
s2 = ((y - media) ** 2).sum() / (n - 1)
num = n * (x * y).sum() - x.sum() * y.sum()
b = num / (n * (x * x).sum() - x.sum() ** 2)
a = y.mean() - b * x.mean()
r = num / math.sqrt((n * (x * x).sum() - x.sum() ** 2) * (n * (y * y).sum() - y.sum() ** 2))
lr = stats.linregress(x, y)
assert abs(b - lr.slope) < 1e-9 and abs(r - lr.rvalue) < 1e-12
resultados = {
    "media": media, "mediana": float(np.median(y)), "moda": float(pd.Series(y).mode().iloc[0]),
    "Q1": cuartil(1), "Q2": cuartil(2), "Q3": cuartil(3), "varianza": s2, "desviacion": math.sqrt(s2),
    "cv_%": math.sqrt(s2) / media * 100, "parametro_mu": el.VAL_CFM.mean(), "b": b, "a": a, "r": r, "R2": r * r,
    "P(banca)": (t.banca == "Sí").mean(),
    "P(banca | acepta)": ((t.banca == "Sí") & (t.acepta_mensajes == "Sí")).sum() / (t.acepta_mensajes == "Sí").sum(),
    "P(3,3)": math.perm(3, 3), "P(30,3)": math.perm(30, 3), "C(30,3)": math.comb(30, 3),
    "P(al menos 1 con banca en 3)": 1 - math.comb(27, 3) / math.comb(30, 3),
}
pd.Series(resultados).round(4)

media                           3.656627e+04
mediana                         4.018500e+04
moda                            4.018500e+04
Q1                              3.413375e+04
Q2                              4.018500e+04
Q3                              4.018500e+04
varianza                        7.036387e+07
desviacion                      8.388318e+03
cv_%                            2.294000e+01
parametro_mu                    3.927645e+04
b                              -2.642171e+02
a                               3.901468e+04
r                              -8.308000e-01
R2                              6.902000e-01
P(banca)                        1.000000e-01
P(banca | acepta)               6.667000e-01
P(3,3)                          6.000000e+00
P(30,3)                         2.436000e+04
C(30,3)                         4.060000e+03
P(al menos 1 con banca en 3)    2.796000e-01
dtype: float64